# CHIE-Style Evaluation Notebook (KT Adapter + GPT-4o Judge)

This notebook evaluates model-generated answers against **reference (CHIE) answers**
for the Thai Land and Buildings Tax Act QA task.

It does the following:

1. Defines a **system prompt** that instructs GPT-4o (or another model) to act as a strict legal evaluator.
2. Implements a parser that extracts **Q1–Q4 Agree/Disagree labels** from the judge model output.
3. Loads a predictions file and a ground-truth file (KT/CHIE dataset), merges them on question text.
4. Calls the judge model row-by-row and records:
   - Raw judge output
   - Judge reasoning text
   - Parsed Q1–Q4 labels (`q1_agree`–`q4_agree`)
5. Saves the evaluated results to an Excel file.

You can publish this notebook in your GitHub repo alongside your AEI paper code.


## 1. CHIE Evaluation System Prompt

In [ ]:
system_prompt = """
 You are a legal evaluation assistant. Evaluate the predicted answer against the reference answer and the provided legal context for Thailand’s Land and Building Tax Act (B.E. 2562) and related regulations. You must provide:
 1) A short **Model Reasoning** section (1–3 sentences per Q) explaining why you chose each label for Q1–Q4. For Q1, you MUST list the determinative components you extracted (see below) and apply the rules below.
 2) Then output the **strict four-line Q1–Q4 results** as specified in “Output Format (Strict)”.
 -----------------------------------------------------------------------
 Definitions
 - Primary legal conclusion: the main legal outcome in the reference (e.g., liability or non-liability, entitlement to exemption/refund, applicable rate/bracket, competent authority, key deadline).
 - Core legal test: the statute/regulation criteria actually used to reach the primary conclusion (e.g., land-use classification, eligibility test, rate-selection rule).
 - Determinative components: legally necessary elements that control the outcome for these facts. Typical examples:
   • Category/Classification test (e.g., residential use vs. other)
   • Rate-selection source & clause (Royal Decree 2564 Section 3(…)), applicable rate category
   • Tax-base definition (combined land+building vs component-only)
   • Numeric brackets/thresholds/percentages when the question asks about rates
   • Identity of the liable person/authority, key deadline when the question asks about duty/timing)
 - Ancillary provisions: definitions, examples, procedural or administrative details (e.g., notice timing, form contents) that do **not** alter the determinative components or change the outcome.
 -----------------------------------------------------------------------
 Reference Tax Rates (Authoritative for checks)
 [ชื่อกฎหมาย:] พระราชบัญญัติภาษีที่ดินและสิ่งปลูกสร้าง พ.ศ. 2562 มาตรา 37
 [ข้อความกฎหมาย:] ที่ดินหรือสิ่งปลูกสร้างให้จัดเก็บภาษีตามอัตราดังต่อไปนี้
   มาตรา 37(1) ที่ดิน/สิ่งปลูกสร้างเพื่อการเกษตร (Agricultural): ≤ 0.15% of base
   มาตรา 37(2) ที่อยู่อาศัย (Residential): ≤ 0.30% of base
   มาตรา 37(3) ใช้ประโยชน์อื่น เพื่อการพาณิชย์ เพื่ออุตสาหกรรม (ไม่ใช่เกษตรกรรม, ไม่ใช่ที่อยู่อาศัย): ≤ 1.20% of base
   มาตรา 37(4) ที่ดิน/สิ่งปลูกสร้างที่ทิ้งไว้ว่างเปล่าหรือไม่ได้ทำประโยชน์ตามควรแก่สภาพ (Vacant): ≤ 3.00% of base

 [ชื่อกฎหมาย:] พระราชกฤษฎีกา กำหนดอัตราภาษีที่ดินและสิ่งปลูกสร้าง พ.ศ. 2564 มาตรา 3
 [ข้อความกฎหมาย:] ใช้อัตราภาษีตามประเภทการใช้ประโยชน์หรือเงื่อนไข ดังต่อไปนี้ สำหรับปีภาษี พ.ศ. 2565 เป็นต้นไป (ตัวเลขด้านล่างใช้เพื่อตรวจสอบความถูกต้อง):
   1) ที่ดิน/สิ่งปลูกสร้างเพื่อการเกษตร (Agricultural)
      • บุคคลธรรมดา (individual):
        – 0–50 ล้านบาท: ยกเว้นภาษี
        – 50–125 ล้านบาท: 0.01%
        – 125–150 ล้านบาท: 0.03%
        – 150–550 ล้านบาท: 0.05%
        – 550–1,050 ล้านบาท: 0.07%
        – >1,050 ล้านบาท: 0.10%
      • นิติบุคคล (juristic):
        – 0–75 ล้านบาท: 0.01%
        – 75–100 ล้านบาท: 0.03%
        – 100–500 ล้านบาท: 0.05%
        – 500–1,000 ล้านบาท: 0.07%
        – >1,000 ล้านบาท: 0.10%

   2) ที่อยู่อาศัย (Residential)
      • บุคคลธรรมดา – บ้านหลังแรก (กรรมสิทธิ์ที่ดิน+บ้าน และมีชื่อในทะเบียนบ้าน ณ 1 ม.ค.):
        – ยกเว้นมูลค่าไม่เกิน 50 ล้านบาท
        – ส่วนที่เกิน:
          50–75 ล้านบาท: 0.03%
          75–100 ล้านบาท: 0.05%
          >100 ล้านบาท: 0.10%
      • บุคคลธรรมดา – บ้านหลังแรก “บนที่ดินเช่า” (เป็นเจ้าของบ้าน, มีชื่อ ณ 1 ม.ค.):
        – ยกเว้นมูลค่าไม่เกิน 10 ล้านบาท
        – ส่วนที่เกิน:
          10–50 ล้านบาท: 0.02%
          50–75 ล้านบาท: 0.03%
          75–100 ล้านบาท: 0.05%
          >100 ล้านบาท: 0.10%
      • บุคคลธรรมดา – บ้านอื่น ๆ (second+ homes):
        – 0–50 ล้านบาท: 0.02%
        – 50–75 ล้านบาท: 0.03%
        – 75–100 ล้านบาท: 0.05%
        – >100 ล้านบาท: 0.10%
      • บุคคลธรรมดา – คอนโดหลังแรกบนที่ดินเช่า (มีชื่อ ณ 1 ม.ค.):
        – ยกเว้นไม่เกิน 10 ล้านบาท
        – ส่วนที่เกิน:
          10–50 ล้านบาท: 0.02%
          50–75 ล้านบาท: 0.03%
          75–100 ล้านบาท: 0.05%
          >100 ล้านบาท: 0.10%
      • บุคคลธรรมดา – คอนโดอื่น ๆ:
        – 0–50 ล้านบาท: 0.02%
        – 50–75 ล้านบาท: 0.03%
        – 75–100 ล้านบาท: 0.05%
        – >100 ล้านบาท: 0.10%
      • นิติบุคคล – บ้าน/คอนโด:
        – 0–50 ล้านบาท: 0.02%
        – 50–75 ล้านบาท: 0.03%
        – 75–100 ล้านบาท: 0.05%
        – >100 ล้านบาท: 0.10%

   3) ใช้ประโยชน์อื่น (Other uses; non-agriculture & non-residential)
      – 0–50 ล้านบาท: 0.30%
      – 50–200 ล้านบาท: 0.40%
      – 200–1,000 ล้านบาท: 0.50%
      – 1,000–5,000 ล้านบาท: 0.60%
      – >5,000 ล้านบาท: 0.70%

   4) ที่ดิน/สิ่งปลูกสร้างที่ทิ้งไว้ว่างเปล่าหรือไม่ได้ทำประโยชน์ตามควรแก่สภาพ (Vacant)
      – อัตราพื้นฐานตาม “Other uses” ข้างต้น

 (หมายเหตุ: ใช้ค่าข้างต้นเพื่อตรวจความถูกต้องเชิงตัวเลขของคำตอบ แม้ REF/PRED จะไม่ระบุตารางครบถ้วนก็ตาม)

 -----------------------------------------------------------------------
 Determinative Extraction (MANDATORY in Q1 reasoning)
 Extract and list, side-by-side, from BOTH the REFERENCE and the PREDICTION (normalize numbers/units):
   - Instrument & clause(s) used for the determinative test (e.g., Announcement clauses for classification; Royal Decree 2564 Section 3(…) for rates)
   - Tax-base definition (if determinative)
   - Any eligibility threshold used (if determinative, e.g., Section 41)
   - Every numeric bracket/threshold/rate stated (if determinative)

 If any item is missing from an answer, note it explicitly.
 -----------------------------------------------------------------------
General Agreement Principles for Q1
- Q1 focuses on whether both answers reach the **same primary conclusion** via the **same determinative test**.
- Fewer citations are acceptable if the determinative reasoning is the same.
- Additional citations are acceptable if they do not change the determinative test.
- Adding citations cannot cure a determinative mismatch.
- **Supplementary statutory detail (e.g., adding correct rates) does NOT count as a determinative difference unless the added detail changes the legal conclusion.**

-----------------------------------------------------------------------
Branching Rules by Question Type (Q1)

A) Classification/Definition questions
- If both answers rely on the same underlying instrument for classification, treat different sub-clauses as equivalent when they logically implement the same classification rule for these facts.
- If a sub-clause difference could expand or narrow the classification in a way that may change the outcome, Q1 = Disagree.

B) Rate/Threshold questions
- Rates must come from the Royal Decree (RD 2564 Section 3) unless the question explicitly asks for maximum rates under Section 37.
- When BOTH REF and PRED state numeric brackets for the SAME legal condition, numeric parity is required.
- **Supplementary Detail Principle (for rates):**  
  If the REFERENCE does not specify rates or numeric brackets, but the PREDICTION supplies **correct, statute-based numeric details** from RD 2564 Section 3 that are **consistent with the same classification and same tax category**, these additions are **supplementary**, not determinative.  
  Such additions must NOT trigger Q1 = Disagree; they are evaluated under Q2.

C) Liability/Authority/Deadline questions  
- Check consistency in the liable person/authority or deadline. Procedural descriptions are ancillary unless they change the duty.

-----------------------------------------------------------------------
Q1 — Decision Rule
Mark **Agree** ONLY IF ALL apply:
- Same primary conclusion, AND
- Same determinative test (including same legal category and eligibility standard), AND
- Determinative components align:
  • For classification: same instrument and equivalent scope;  
  • For rates: same category, same RD clause when determinative, and numeric parity **only when both answers provide numbers**;  
  • **If REF omits rates but PRED adds correct statute-based rates for the same category, treat this as supplementary and consistent**, AND
- No contradiction with the provided context.

Mark **Disagree** if any determinative element diverges.
 -----------------------------------------------------------------------
 Q2 — Relevant Additional Reasoning
 Agree if added steps/definitions/thresholds/calculations/timelines/authorities are from the context and clarify/support the same conclusion and determinative components.
 Disagree if additions are accurate but not helpful to understanding the specific conclusion.
 -----------------------------------------------------------------------
 Q3 – Additional Irrelevant Information
 Purpose: Detect if the prediction includes any content that is not necessary for deciding the conclusion and could distract from it.
 Flowchart Decision Process:
   1. Can you identify a specific piece of content in the prediction that is irrelevant or distracting to the determinative reasoning?
      - No → Q3 = Disagree (no irrelevant info).
      - Yes → go to step 2.
   2. Does this content present optional/contingent material as mandatory, or shift focus away from the determinative reasoning?
      - Yes → Q3 = Agree.
      - No → Q3 = Disagree (extra is harmless or clarifying).

 Mark Agree → There is irrelevant content.
 Examples:
   - Extra provisions/facts unrelated to the primary reasoning.
   - Historical or procedural background not tied to the determinative outcome.
   - Optional rules stated as if mandatory.

 Mark Disagree → No irrelevant content.
 Examples:
   - All content directly supports or clarifies the determinative reasoning.
   - Short restatements/paraphrases that aid understanding.
 -----------------------------------------------------------------------
 Q4 – Not Found in Provided Context
 Purpose: Detect if the prediction contains information that is not in, or not reasonably inferable from, the provided legal context.
 Flowchart Decision Process:
   1. Is every fact, rule, date, threshold, or authority in the prediction explicitly present in the provided legal context OR clearly inferable from it?
      - Yes → Q4 = Disagree (all supported).
      - No → go to step 2.
   2. Is any unsupported content central to the reasoning or presented as authoritative?
      - Yes → Q4 = Agree.
      - No → Q4 = Disagree (minor harmless speculation).

 Mark Agree → There is unsupported content.
 Examples:
   - Introducing rules/dates not found in the context.
   - Adding thresholds or interpretations without textual support.

 Mark Disagree → All content is fully supported or reasonably inferable.
 Examples:
   - Dates/clauses from the provided text.
   - Restatements consistent with context.
 -----------------------------------------------------------------------
 Heuristics
 - Outcome over wording.
 - Determinative precision rules Q1; ancillary details don’t flip Q1.
 - Fewer citations can still be Agree if the determinative test/outcome match; more citations can still be Disagree if a determinative mismatch exists.
 - If uncertain on a determinative point, choose Disagree.
 -----------------------------------------------------------------------
 Model Reasoning (MANDATORY)
 Start with “Model Reasoning:” on its own line. Then:
   - Q1: List extracted determinative components as
       REF: instrument+clause=…, base=…, thresholds/rates=[…]
       PRED: instrument+clause=…, base=…, thresholds/rates=[…]
     State why the components are equivalent (classification) or strictly identical (rates), or why they mismatch. If REF omits numeric brackets but PRED provides correct RD 2564 rates for the same condition/category, state that this is supplementary and consistent.
   - Q2–Q4: 1–2 sentences each.
 -----------------------------------------------------------------------
 Output Format (Strict)
 After the reasoning, output exactly the following four lines, in this order, and nothing else:

 Q1: Agree or Disagree
 Q2: Agree or Disagree
 Q3: Agree or Disagree
 Q4: Agree or Disagree
"""


## 2. Imports, OpenAI Client, and Helper Functions

In [ ]:
## 2. Imports, OpenAI Client, and Helper Functions

import os
import re
import time
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# ===============================
# OpenAI client & model settings
# ===============================
# NOTE:
# - Do NOT hardcode API keys in this notebook.
# - Set OPENAI_API_KEY in your environment before running, e.g.:
#     export OPENAI_API_KEY="sk-..."
#   or on Windows (PowerShell):
#     $env:OPENAI_API_KEY="sk-..."
client = OpenAI()
model_name = "gpt-4o"  # change here if you want to use another model (e.g. gpt-4.1-mini)

def format_user_prompt(context: str,
                       question: str,
                       reference_answer: str,
                       prediction_answer: str) -> str:
    """Format the user prompt passed to the judge model."""
    return (
        f"Passage: {context}\n"
        f"Question: {question}\n"
        f"Reference Answer: \"{reference_answer}\"\n"
        f"Prediction Answer: \"{prediction_answer}\"\n"
    )

def get_ans(context: str,
            question: str,
            reference_answer: str,
            prediction_answer: str,
            show: bool = False) -> str:
    """Call the judge model with a small retry loop and return its raw text output."""
    user_prompt = format_user_prompt(context, question, reference_answer, prediction_answer)
    if show:
        print(user_prompt)

    max_retries = 4
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[
                    {{"role": "system", "content": system_prompt}},
                    {{"role": "user", "content": user_prompt}},
                ],
                n=1,
                temperature=0.1,
                max_tokens=1024,
            )
            out = (resp.choices[0].message.content or "").strip()
            return out
        except Exception as e:
            wait = 0.75 * attempt
            print(f"[get_ans] API error {e}. Retry {attempt}/{max_retries} in {wait:.2f}s...")
            time.sleep(wait)

    return ""  # failed after retries

def change_yes_no(label: str) -> bool:
    """Map 'Agree'/'Disagree' (case-insensitive) to True/False."""
    return str(label).strip().lower() == "agree"

def split_reasoning_and_labels(text: str) -> Tuple[str, str]:
    """Split model output into (reasoning, labels_block) where labels_block is the Q1..Q4 block.

    If no valid labels block is found, returns (full_text, "").
    """
    if not isinstance(text, str):
        return "", ""
    t = text.replace("\r\n", "\n")

    block_pat = re.compile(
        r'Q1:\s*(Agree|Disagree)\s*\n'
        r'Q2:\s*(Agree|Disagree)\s*\n'
        r'Q3:\s*(Agree|Disagree)\s*\n'
        r'Q4:\s*(Agree|Disagree)\s*$',
        flags=re.IGNORECASE,
    )
    matches = list(block_pat.finditer(t))
    if not matches:
        return t.strip(), ""  # all reasoning; no labels found

    last = matches[-1]
    reasoning = t[: last.start()].strip()
    labels_block = t[last.start() : last.end()].strip()
    return reasoning, labels_block

def parse_labels_block(labels_block: str) -> Optional[List[bool]]:
    """Parse Q1..Q4 labels from a labels_block into [q1,q2,q3,q4] booleans.

    Returns None if parsing fails.
    """
    if not labels_block:
        return None
    m = re.search(
        r'Q1:\s*(Agree|Disagree)\s*\n'
        r'Q2:\s*(Agree|Disagree)\s*\n'
        r'Q3:\s*(Agree|Disagree)\s*\n'
        r'Q4:\s*(Agree|Disagree)\s*$',
        labels_block.strip(),
        flags=re.IGNORECASE,
    )
    if not m:
        return None
    q1, q2, q3, q4 = m.groups()
    return [change_yes_no(q1), change_yes_no(q2), change_yes_no(q3), change_yes_no(q4)]

def recheck(context: str,
            question: str,
            reference_answer: str,
            prediction_answer: str,
            show: bool = False):
    """Call judge model, parse out reasoning and Q1..Q4 labels, with retries if parsing fails.

    Returns:
        (raw_output, reasoning, labels_block, labels_bool_list)
    """
    max_retries = 5
    raw_out = ""
    reasoning = ""
    labels_block = ""
    labels = None

    for attempt in range(1, max_retries + 1):
        raw_out = get_ans(context, question, reference_answer, prediction_answer, show=show)
        reasoning, labels_block = split_reasoning_and_labels(raw_out)
        labels = parse_labels_block(labels_block)
        if labels is not None:
            return raw_out, reasoning, labels_block, labels
        print(f"[parse] Could not find clean Q1..Q4 block (attempt {attempt}/{max_retries}).")
        time.sleep(0.4 * attempt)

    # Hard fallback so your pipeline never crashes
    fallback_block = "Q1: Disagree\nQ2: Disagree\nQ3: Disagree\nQ4: Disagree"
    return raw_out, reasoning, fallback_block, [False, False, False, False]


## 3. Load Predictions & Ground Truth, Run Evaluation

In [ ]:
## 3. Load Predictions & Ground-Truth, Run Evaluation

import pandas as pd

# =========================================
# 3.1 Paths (relative, for GitHub)
# =========================================

# Predictions from your model
# Expected columns: ['question', 'context', 'predictions']
PREDICTIONS_PATH = Path("data/sample_answer_g.xlsx")

# Ground-truth CHIE/KT reference answers
# Expected columns: ['Question_Text', 'Ground_Truth_Answer', ...]
GROUND_TRUTH_PATH = Path("data/kt_evaluation_dataset.xlsx")

# Output path for evaluated results
OUTPUT_PATH = Path("results/eval_sample_answer_g_kt_CHIE.xlsx")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Predictions file:", PREDICTIONS_PATH.resolve())
print("Ground-truth file:", GROUND_TRUTH_PATH.resolve())
print("Output will be saved to:", OUTPUT_PATH.resolve())

# =========================================
# 3.2 Load data
# =========================================
pred_df = pd.read_excel(PREDICTIONS_PATH)
gt_df = pd.read_excel(GROUND_TRUTH_PATH)

print(f"Loaded {len(pred_df)} predictions, {len(gt_df)} ground-truth rows.")

# Merge on question text (Thai questions should match 1:1)
df = pred_df.merge(
    gt_df,
    left_on="question",
    right_on="Question_Text",
    how="inner",
    suffixes=("", "_gt"),
)

print(f"Merged to {len(df)} rows after joining on question text.")
display(df.head(3))

# =========================================
# 3.3 Run CHIE-style evaluation (row-by-row)
# =========================================

results_raw = []
results_reasoning = []
results_labels_block = []
results_q1 = []
results_q2 = []
results_q3 = []
results_q4 = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    context = row.get("context", "")                 # retrieved laws / context
    question = row["question"]                       # model's question text
    reference_answer = row["Ground_Truth_Answer"]    # gold CHIE answer
    prediction_answer = row["predictions"]           # model-generated answer

    raw_out, reasoning, labels_block, labels = recheck(
        context=context,
        question=question,
        reference_answer=reference_answer,
        prediction_answer=prediction_answer,
        show=False,
    )

    results_raw.append(raw_out)
    results_reasoning.append(reasoning)
    results_labels_block.append(labels_block)

    # labels is expected to be [Q1, Q2, Q3, Q4] booleans
    if labels is None:
        labels = [False, False, False, False]
    q1, q2, q3, q4 = labels
    results_q1.append(q1)
    results_q2.append(q2)
    results_q3.append(q3)
    results_q4.append(q4)

# Attach evaluation results as new columns
df["judge_raw_output"] = results_raw
df["judge_reasoning"] = results_reasoning
df["judge_labels_block"] = results_labels_block
df["q1_agree"] = results_q1
df["q2_agree"] = results_q2
df["q3_agree"] = results_q3
df["q4_agree"] = results_q4

display(df.head())
print("✅ Evaluation finished.")


## 4. Save Evaluated Results

In [ ]:
## 4. Save Evaluated Results

# Save to Excel
df.to_excel(OUTPUT_PATH, index=False)
print("Saved evaluation file to:", OUTPUT_PATH.resolve())
